In [8]:
# =============================================================================
# PHASE 1A: INVESTIGATE TEXT TRUNCATION IN V2.5
# =============================================================================

import pandas as pd
from src.config import OUTPUTS_DIR

# Load V2.5 results
df_v25 = pd.read_excel(OUTPUTS_DIR / "ledger_transcription_v2.5_latest.xlsx")

print("="*70)
print("TEXT TRUNCATION INVESTIGATION")
print("="*70)

# 1. Basic stats on description length
print("\n1. DESCRIPTION LENGTH STATISTICS")
print("-"*50)
df_v25['desc_length'] = df_v25['description'].astype(str).apply(len)
print(df_v25['desc_length'].describe())

# 2. Find descriptions that might be truncated
# Indicators: ends abruptly, very short, ends mid-word
print("\n2. POTENTIALLY TRUNCATED DESCRIPTIONS")
print("-"*50)

# Short descriptions (excluding section headers and titles which are naturally short)
entry_rows = df_v25[df_v25['row_type'] == 'entry'].copy()
short_entries = entry_rows[entry_rows['desc_length'] < 10]
print(f"Entry rows with very short descriptions (<10 chars): {len(short_entries)}")

# Descriptions ending with unusual patterns (might indicate truncation)
import re
def looks_truncated(desc):
    desc = str(desc).strip()
    if len(desc) < 3:
        return False
    # Ends with lowercase letter followed by nothing (mid-word?)
    if re.search(r'[a-z]$', desc) and not desc.endswith(('de', 'ye', 'the', 'for', 'and')):
        return True
    # Ends with hyphen or dash
    if desc.endswith('-') or desc.endswith('–'):
        return True
    return False

potentially_truncated = df_v25[df_v25['description'].apply(looks_truncated)]
print(f"Descriptions that might be truncated: {len(potentially_truncated)}")

# 3. Sample of potentially truncated entries for manual review
print("\n3. SAMPLE FOR MANUAL REVIEW")
print("-"*50)
print("Please compare these against the original PDF images:\n")

sample_size = min(20, len(potentially_truncated))
if sample_size > 0:
    sample = potentially_truncated.sample(sample_size, random_state=42)
    for idx, row in sample.iterrows():
        print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
        print(f"  Description: \"{row['description']}\"")
        print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
        print()
else:
    # If no truncated ones found, show random sample of short descriptions
    print("No obviously truncated descriptions found. Here's a sample of short entries:\n")
    sample = entry_rows.nsmallest(20, 'desc_length')
    for idx, row in sample.iterrows():
        print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
        print(f"  Description: \"{row['description']}\"")
        print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
        print()

# 4. Distribution by file (to see if truncation is file-specific)
print("\n4. AVERAGE DESCRIPTION LENGTH BY FILE")
print("-"*50)
file_stats = df_v25.groupby('file_id')['desc_length'].agg(['mean', 'min', 'max', 'count'])
file_stats = file_stats.sort_values('mean')
print(file_stats.head(10))  # Files with shortest average descriptions

# Save sample for manual review
sample_path = OUTPUTS_DIR / "truncation_investigation_sample.csv"
if sample_size > 0:
    sample.to_csv(sample_path, index=False)
else:
    entry_rows.nsmallest(20, 'desc_length').to_csv(sample_path, index=False)
print(f"\n✓ Saved sample for manual review: {sample_path}")

print("\n" + "="*70)
print("NEXT STEP: Please review the sample above against the original PDFs")
print("and let me know what patterns you see in the truncation.")
print("="*70)

TEXT TRUNCATION INVESTIGATION

1. DESCRIPTION LENGTH STATISTICS
--------------------------------------------------
count    7592.000000
mean       22.293994
std        15.496226
min         1.000000
25%        12.000000
50%        19.000000
75%        29.000000
max       231.000000
Name: desc_length, dtype: float64

2. POTENTIALLY TRUNCATED DESCRIPTIONS
--------------------------------------------------
Entry rows with very short descriptions (<10 chars): 791
Descriptions that might be truncated: 5891

3. SAMPLE FOR MANUAL REVIEW
--------------------------------------------------
Please compare these against the original PDF images:

File: 1881, Page: 7, Row: 30
  Description: "Straw"
  Amounts: £4.0/10.0/0

File: 1889, Page: 2, Row: 43
  Description: "Dr Richards Prize Fund"
  Amounts: £nan/nan/nan

File: 1873, Page: 11, Row: 32
  Description: "Yamton"
  Amounts: £3.0/3.0/0

File: 1895, Page: 2, Row: 13
  Description: "Rates, Taxes & Insurance"
  Amounts: £530.0/13.0/2

File: 1860, Pa

In [11]:
# =============================================================================
# PHASE 1A (CONTINUED): INVESTIGATE LONG DESCRIPTIONS
# =============================================================================

import pandas as pd
from src.config import OUTPUTS_DIR

# Load V2.5 results
df_v25 = pd.read_excel(OUTPUTS_DIR / "ledger_transcription_v2.5_latest.xlsx")
df_v25['desc_length'] = df_v25['description'].astype(str).apply(len)

print("="*70)
print("LONG DESCRIPTION INVESTIGATION")
print("="*70)

# Get entry rows with longer descriptions
entry_rows = df_v25[df_v25['row_type'] == 'entry'].copy()

# Sample of longest descriptions
print("\n1. SAMPLE OF LONGEST DESCRIPTIONS")
print("-"*50)
print("Please compare these against the original PDF images:\n")

longest = entry_rows.nlargest(15, 'desc_length')
for idx, row in longest.iterrows():
    print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
    print(f"  Length: {row['desc_length']} chars")
    print(f"  Description: \"{row['description'][:100]}{'...' if row['desc_length'] > 100 else ''}\"")
    print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
    print()

# Sample of medium-length descriptions (where truncation might be partial)
print("\n2. SAMPLE OF MEDIUM-LENGTH DESCRIPTIONS (50-80 chars)")
print("-"*50)
medium = entry_rows[(entry_rows['desc_length'] >= 50) & (entry_rows['desc_length'] <= 80)]
sample_medium = medium.sample(min(15, len(medium)), random_state=42)

for idx, row in sample_medium.iterrows():
    print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
    print(f"  Length: {row['desc_length']} chars")
    print(f"  Description: \"{row['description']}\"")
    print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
    print()

# Check 1889.pdf specifically
print("\n3. ANALYSIS OF 1889.PDF (DIFFERENT STRUCTURE)")
print("-"*50)
df_1889 = df_v25[df_v25['file_id'] == '1889']
print(f"Total rows in 1889.pdf: {len(df_1889)}")
print(f"Pages: {df_1889['page_number'].nunique()}")
print(f"Row types: {df_1889['row_type'].value_counts().to_dict()}")
print(f"\nSample rows from 1889.pdf:")
for idx, row in df_1889.head(10).iterrows():
    print(f"  Page {row['page_number']}, Row {row.get('row_index', 'N/A')}: \"{row['description'][:50]}...\" | £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")

# Save samples for manual review
longest.to_csv(OUTPUTS_DIR / "long_descriptions_sample.csv", index=False)
sample_medium.to_csv(OUTPUTS_DIR / "medium_descriptions_sample.csv", index=False)
df_1889.to_csv(OUTPUTS_DIR / "file_1889_analysis.csv", index=False)

print(f"\n✓ Saved: long_descriptions_sample.csv")
print(f"✓ Saved: medium_descriptions_sample.csv")
print(f"✓ Saved: file_1889_analysis.csv")

print("\n" + "="*70)
print("NEXT STEP: Review the long/medium descriptions above against PDFs")
print("Also, let's look at 1889.pdf structure to design a specific prompt")
print("="*70)

LONG DESCRIPTION INVESTIGATION

1. SAMPLE OF LONGEST DESCRIPTIONS
--------------------------------------------------
Please compare these against the original PDF images:

File: 1753, Page: 1, Row: 35
  Length: 231 chars
  Description: "Received of the Administratrix of Dr Edgcumbe & of hands of Mr John Edgcumbe the sum of Ninety two P..."
  Amounts: £92.0/0.0/0

File: 1855, Page: 3, Row: 22
  Length: 185 chars
  Description: "Mistakes in last years Act the sum having been charged for drawing on for clergys Dam. whereas a che..."
  Amounts: £38.0/nan/10

File: 1753, Page: 3, Row: 33
  Length: 175 chars
  Description: "Pd Mr. Townsend in part of his Bill of Work done in the Chapple by Order of Dr. Edgcumbe (being in a..."
  Amounts: £92.0/nan/nan

File: 1731, Page: 4, Row: 4
  Length: 152 chars
  Description: "Paid Mr Wentworth by Consent a gratuity for his great Care and Pains in detecting One Cooks Boy Anno..."
  Amounts: £4.0/4.0/0

File: 1738, Page: 4, Row: 4
  Length: 139 chars
  D

In [12]:
# =============================================================================
# PHASE 1A (CONTINUED): INVESTIGATE LONG DESCRIPTIONS - FULL TEXT
# =============================================================================

import pandas as pd
from src.config import OUTPUTS_DIR

# Load V2.5 results
df_v25 = pd.read_excel(OUTPUTS_DIR / "ledger_transcription_v2.5_latest.xlsx")
df_v25['desc_length'] = df_v25['description'].astype(str).apply(len)

print("="*70)
print("LONG DESCRIPTION INVESTIGATION (FULL TEXT)")
print("="*70)

# Get entry rows with longer descriptions
entry_rows = df_v25[df_v25['row_type'] == 'entry'].copy()

# Sample of longest descriptions
print("\n1. SAMPLE OF LONGEST DESCRIPTIONS")
print("-"*70)
print("Please compare these against the original PDF images:\n")

longest = entry_rows.nlargest(15, 'desc_length')
for idx, row in longest.iterrows():
    print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
    print(f"  Length: {row['desc_length']} chars")
    print(f"  Description: \"{row['description']}\"")
    print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
    print("-"*70)
    print()

# Sample of medium-length descriptions (where truncation might be partial)
print("\n2. SAMPLE OF MEDIUM-LENGTH DESCRIPTIONS (50-80 chars)")
print("-"*70)
medium = entry_rows[(entry_rows['desc_length'] >= 50) & (entry_rows['desc_length'] <= 80)]
sample_medium = medium.sample(min(15, len(medium)), random_state=42)

for idx, row in sample_medium.iterrows():
    print(f"File: {row['file_id']}, Page: {row['page_number']}, Row: {row.get('row_index', 'N/A')}")
    print(f"  Length: {row['desc_length']} chars")
    print(f"  Description: \"{row['description']}\"")
    print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
    print("-"*70)
    print()

# Check 1889.pdf specifically
print("\n3. ANALYSIS OF 1889.PDF (DIFFERENT STRUCTURE)")
print("-"*70)
df_1889 = df_v25[df_v25['file_id'] == '1889']
print(f"Total rows in 1889.pdf: {len(df_1889)}")
print(f"Pages: {df_1889['page_number'].nunique()}")
print(f"Row types: {df_1889['row_type'].value_counts().to_dict()}")
print(f"\nAll rows from 1889.pdf:")
print("-"*70)
for idx, row in df_1889.iterrows():
    print(f"Page {row['page_number']}, Row {row.get('row_index', 'N/A')}:")
    print(f"  Type: {row['row_type']}")
    print(f"  Description: \"{row['description']}\"")
    print(f"  Amounts: £{row['amount_pounds']}/{row['amount_shillings']}/{row['amount_pence_whole']}")
    print()

# Save samples for manual review
longest.to_csv(OUTPUTS_DIR / "long_descriptions_sample.csv", index=False)
sample_medium.to_csv(OUTPUTS_DIR / "medium_descriptions_sample.csv", index=False)
df_1889.to_csv(OUTPUTS_DIR / "file_1889_analysis.csv", index=False)

print(f"\n✓ Saved: long_descriptions_sample.csv")
print(f"✓ Saved: medium_descriptions_sample.csv")
print(f"✓ Saved: file_1889_analysis.csv")

print("\n" + "="*70)
print("NEXT STEP: Review the long/medium descriptions above against PDFs")
print("Also, let's look at 1889.pdf structure to design a specific prompt")
print("="*70)

LONG DESCRIPTION INVESTIGATION (FULL TEXT)

1. SAMPLE OF LONGEST DESCRIPTIONS
----------------------------------------------------------------------
Please compare these against the original PDF images:

File: 1753, Page: 1, Row: 35
  Length: 231 chars
  Description: "Received of the Administratrix of Dr Edgcumbe & of hands of Mr John Edgcumbe the sum of Ninety two Pounds in full for her towards the Discharge of ye Workmen Bills unleyed by Dr Edgcumbe in the Chapple (the whole sum being 184.1.1)"
  Amounts: £92.0/0.0/0
----------------------------------------------------------------------

File: 1855, Page: 3, Row: 22
  Length: 185 chars
  Description: "Mistakes in last years Act the sum having been charged for drawing on for clergys Dam. whereas a cheque was drawn on Gosling Bank for Mr. Sanders div. & now allowed by late Rector Exton"
  Amounts: £38.0/nan/10
----------------------------------------------------------------------

File: 1753, Page: 3, Row: 33
  Length: 175 chars
  Desc

In [17]:
# =============================================================================
# COMPARE COLLEAGUE'S RESULTS VS V2.5 FOR 1700 PAGE 7
# =============================================================================

import pandas as pd
from src.config import OUTPUTS_DIR, DATA_DIR

# Load colleague's results
# UPDATE THIS PATH to where you saved the file on your computer
colleague_path = DATA_DIR.parent / "1700_page7_image.xlsx"  # Try this first
# OR use the full path like:
# colleague_path = "/Users/hamidostadi/Documents/0. Research/H-AI KHu lab/Week 3 Task/1700_page7_image.xlsx"
# OR if you put it in the project folder:
# colleague_path = OUTPUTS_DIR.parent / "1700_page7_image.xlsx"

# Check if file exists
from pathlib import Path
if not Path(colleague_path).exists():
    print(f"ERROR: File not found at: {colleague_path}")
    print("\nPlease update 'colleague_path' to the correct location.")
    print("You can find the file location by checking where you downloaded it.")
else:
    df_colleague = pd.read_excel(colleague_path)

    # Load V2.5 results and filter for 1700 page 7
    df_v25 = pd.read_excel(OUTPUTS_DIR / "ledger_transcription_v2.5_latest.xlsx")
    df_v25_page = df_v25[(df_v25['file_id'] == '1700') & (df_v25['page_number'] == 7)].copy()

    print("="*80)
    print("COMPARISON: COLLEAGUE vs V2.5 — File: 1700, Page: 7")
    print("="*80)

    # Basic stats
    print("\n1. BASIC STATISTICS")
    print("-"*80)
    print(f"Colleague's extraction: {len(df_colleague)} rows")
    print(f"V2.5 extraction: {len(df_v25_page)} rows")

    # Show colleague's columns
    print(f"\nColleague's columns: {list(df_colleague.columns)}")

    # Show both datasets side by side
    print("\n2. COLLEAGUE'S RESULTS (ALL ROWS)")
    print("-"*80)
    for idx, row in df_colleague.iterrows():
        desc = row.get('description', row.get('Description', 'N/A'))
        pounds = row.get('amount_pounds', row.get('Pounds', row.get('£', 'N/A')))
        shillings = row.get('amount_shillings', row.get('Shillings', row.get('s', 'N/A')))
        pence = row.get('amount_pence_whole', row.get('Pence', row.get('d', 'N/A')))
        row_type = row.get('row_type', row.get('Row Type', 'N/A'))
        confidence = row.get('confidence_score', row.get('Confidence', 'N/A'))
        
        print(f"Row {idx+1}: [{row_type}]")
        print(f"  Description: \"{desc}\"")
        print(f"  Amounts: £{pounds} / {shillings}s / {pence}d | Confidence: {confidence}")
        print()

    print("\n3. V2.5 RESULTS (ALL ROWS)")
    print("-"*80)
    for idx, row in df_v25_page.iterrows():
        print(f"Row {row.get('row_index', idx+1)}: [{row['row_type']}]")
        print(f"  Description: \"{row['description']}\"")
        print(f"  Amounts: £{row['amount_pounds']} / {row['amount_shillings']}s / {row['amount_pence_whole']}d | Confidence: {row['confidence_score']}")
        print()

    # Side-by-side comparison (if row counts match or are close)
    print("\n4. SIDE-BY-SIDE COMPARISON")
    print("-"*80)
    print(f"{'Row':<5} {'Colleague £/s/d':<20} {'V2.5 £/s/d':<20} {'Match?':<10}")
    print("-"*80)

    # Try to align by row index
    n_compare = min(len(df_colleague), len(df_v25_page))
    matches = 0

    for i in range(n_compare):
        # Colleague values
        c_row = df_colleague.iloc[i]
        c_pounds = c_row.get('amount_pounds', c_row.get('Pounds', c_row.get('£', '')))
        c_shillings = c_row.get('amount_shillings', c_row.get('Shillings', c_row.get('s', '')))
        c_pence = c_row.get('amount_pence_whole', c_row.get('Pence', c_row.get('d', '')))
        
        # V2.5 values
        v_row = df_v25_page.iloc[i]
        v_pounds = v_row['amount_pounds']
        v_shillings = v_row['amount_shillings']
        v_pence = v_row['amount_pence_whole']
        
        # Format for display
        c_str = f"{c_pounds}/{c_shillings}/{c_pence}"
        v_str = f"{v_pounds}/{v_shillings}/{v_pence}"
        
        # Check match (handle NaN and type differences)
        def normalize(val):
            if pd.isna(val) or str(val).lower() == 'nan' or val == '':
                return None
            try:
                return float(val)
            except:
                return str(val)
        
        match = (normalize(c_pounds) == normalize(v_pounds) and 
                 normalize(c_shillings) == normalize(v_shillings) and 
                 normalize(c_pence) == normalize(v_pence))
        
        if match:
            matches += 1
        
        match_str = "✓" if match else "✗"
        print(f"{i+1:<5} {c_str:<20} {v_str:<20} {match_str:<10}")

    print("-"*80)
    print(f"Matching rows: {matches}/{n_compare} ({matches/n_compare*100:.1f}%)")

    # Currency validation comparison
    print("\n5. CURRENCY VALIDATION CHECK")
    print("-"*80)

    def check_currency_violations(df, name):
        violations = 0
        for idx, row in df.iterrows():
            shillings = row.get('amount_shillings', row.get('Shillings', row.get('s', None)))
            pence = row.get('amount_pence_whole', row.get('Pence', row.get('d', None)))
            
            try:
                if pd.notna(shillings) and float(shillings) >= 20:
                    violations += 1
            except:
                pass
            try:
                if pd.notna(pence) and float(pence) >= 12:
                    violations += 1
            except:
                pass
        
        print(f"{name}: {violations} currency violations")
        return violations

    check_currency_violations(df_colleague, "Colleague")
    check_currency_violations(df_v25_page, "V2.5")

    # Save comparison for manual review
    comparison_path = OUTPUTS_DIR / "comparison_colleague_vs_v25_1700p7.xlsx"
    with pd.ExcelWriter(comparison_path) as writer:
        df_colleague.to_excel(writer, sheet_name='Colleague', index=False)
        df_v25_page.to_excel(writer, sheet_name='V2.5', index=False)

    print(f"\n✓ Saved comparison file: {comparison_path}")

    print("\n" + "="*80)
    print("NEXT STEP: Compare both outputs against the original 1700.pdf Page 7")
    print("="*80)

COMPARISON: COLLEAGUE vs V2.5 — File: 1700, Page: 7

1. BASIC STATISTICS
--------------------------------------------------------------------------------
Colleague's extraction: 41 rows
V2.5 extraction: 0 rows

Colleague's columns: ['file_id', 'page_number', 'row_index', 'row_type', 'description', 'amount_pounds', 'amount_shillings', 'amount_pence_whole', 'amount_pence_fraction', 'group_brace_id', 'notes']

2. COLLEAGUE'S RESULTS (ALL ROWS)
--------------------------------------------------------------------------------
Row 1: [entry]
  Description: ""The quitrents Ann.""
  Amounts: £2.0 / 15.0s / 7.0d | Confidence: N/A

Row 2: [entry]
  Description: ""Ralph Marshalls capons""
  Amounts: £nan / nans / nand | Confidence: N/A

Row 3: [entry]
  Description: ""Lord Ann. 2. 0. 10. mich. 1. 18. 1.""
  Amounts: £3.0 / 19.0s / 2.0d | Confidence: N/A

Row 4: [entry]
  Description: ""Calcott Ann""
  Amounts: £1.0 / 10.0s / 0.0d | Confidence: N/A

Row 5: [entry]
  Description: ""Cudlington mills 

ZeroDivisionError: division by zero

In [18]:
# =============================================================================
# COMPARE COLLEAGUE'S RESULTS VS V2.5 FOR 1700 PAGE 7 (WITH DEBUG)
# =============================================================================

import pandas as pd
from src.config import OUTPUTS_DIR, DATA_DIR

# Load colleague's results
colleague_path = DATA_DIR.parent / "1700_page7_image.xlsx"

from pathlib import Path
if not Path(colleague_path).exists():
    print(f"ERROR: File not found at: {colleague_path}")
    print("\nPlease update 'colleague_path' to the correct location.")
else:
    df_colleague = pd.read_excel(colleague_path)

    # Load V2.5 results
    df_v25 = pd.read_excel(OUTPUTS_DIR / "ledger_transcription_v2.5_latest.xlsx")
    
    # DEBUG: Check what file_ids exist in V2.5
    print("="*80)
    print("DEBUG: CHECKING DATA")
    print("="*80)
    print(f"\nColleague's file loaded: {len(df_colleague)} rows")
    print(f"Colleague's columns: {list(df_colleague.columns)}")
    print(f"\nFirst few rows of colleague's data:")
    print(df_colleague.head())
    
    print(f"\n\nV2.5 total rows: {len(df_v25)}")
    print(f"\nUnique file_ids in V2.5: {sorted(df_v25['file_id'].unique())}")
    
    # Check if '1700' exists (might be stored differently)
    print(f"\nSearching for '1700' in file_id...")
    matches_1700 = df_v25[df_v25['file_id'].astype(str).str.contains('1700')]
    print(f"Rows containing '1700': {len(matches_1700)}")
    
    if len(matches_1700) > 0:
        print(f"Pages available for 1700: {sorted(matches_1700['page_number'].unique())}")
    
    # Try different variations of the file_id
    for file_id_try in ['1700', '1700.pdf', 1700, '1700-1701']:
        df_test = df_v25[df_v25['file_id'] == file_id_try]
        if len(df_test) > 0:
            print(f"\nFound {len(df_test)} rows with file_id = '{file_id_try}'")
            print(f"Pages: {sorted(df_test['page_number'].unique())}")

    print("\n" + "="*80)
    print("Please check the output above to find the correct file_id for 1700")
    print("="*80)

DEBUG: CHECKING DATA

Colleague's file loaded: 41 rows
Colleague's columns: ['file_id', 'page_number', 'row_index', 'row_type', 'description', 'amount_pounds', 'amount_shillings', 'amount_pence_whole', 'amount_pence_fraction', 'group_brace_id', 'notes']

First few rows of colleague's data:
   file_id  page_number  row_index row_type  \
0     1700            7          1    entry   
1     1700            7          2    entry   
2     1700            7          3    entry   
3     1700            7          4    entry   
4     1700            7          5    entry   

                             description  amount_pounds  amount_shillings  \
0                   "The quitrents Ann."            2.0              15.0   
1               "Ralph Marshalls capons"            NaN               NaN   
2  "Lord Ann. 2. 0. 10. mich. 1. 18. 1."            3.0              19.0   
3                          "Calcott Ann"            1.0              10.0   
4     "Cudlington mills Ann. 4. 14. 11." 